# 29th Aug 2026

In [1]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torch import nn,optim
import torch
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
train_dataset = datasets.MNIST(root='../../../data', train=True, download=True, transform=transforms.ToTensor())
test_dataset = datasets.MNIST(root='../../../data', train=False, download=True, transform=transforms.ToTensor())

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

class MNISTPrediction(nn.Module):
    def __init__(self):
        super(MNISTPrediction, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)
        x = self.conv2(x)
        x = self.relu(x)
        x = self.pool2(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x
model = MNISTPrediction()
optimizer = optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()
num_epochs=10
for epoch in range(1,num_epochs+1):
    model.train()
    total_loss = 0
    for batch_idx, (data, target) in enumerate(train_loader):
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch {epoch}/{num_epochs}, Loss: {total_loss/len(train_loader)}')
model.eval()
with torch.no_grad():
    actual = []
    predicted = []
    for data, target in train_loader:
        output = model(data)
        _, pred = torch.max(output.data, 1)
        predicted.extend(pred)
        actual.extend(target)
    print(f'Accuracy: {accuracy_score(actual, predicted)}')
    print(f'Precision: {precision_score(actual, predicted, average="macro")}')
    print(f'Recall: {recall_score(actual, predicted, average="macro")}')    
    print(f'F1 Score: {f1_score(actual, predicted, average="macro")}')

with torch.no_grad():
    actual = []
    predicted = []
    for data, target in test_loader:
        output = model(data)
        _, pred = torch.max(output.data, 1)
        predicted.extend(pred)
        actual.extend(target)
    print(f'Accuracy: {accuracy_score(actual, predicted)}')
    print(f'Precision: {precision_score(actual, predicted, average="macro")}')
    print(f'Recall: {recall_score(actual, predicted, average="macro")}')    
    print(f'F1 Score: {f1_score(actual, predicted, average="macro")}')


100%|██████████| 9.91M/9.91M [00:02<00:00, 3.71MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 125kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.10MB/s]
100%|██████████| 4.54k/4.54k [00:00<?, ?B/s]


Epoch 1/10, Loss: 0.5380986680878378
Epoch 2/10, Loss: 0.15768526663491403
Epoch 3/10, Loss: 0.1027163562150414
Epoch 4/10, Loss: 0.07806317706151542
Epoch 5/10, Loss: 0.06383367322299924
Epoch 6/10, Loss: 0.05525045583857648
Epoch 7/10, Loss: 0.048174150183107826
Epoch 8/10, Loss: 0.0431618388472393
Epoch 9/10, Loss: 0.03907185033046162
Epoch 10/10, Loss: 0.03585078200769188
Accuracy: 0.9909333333333333
Precision: 0.9909215379227249
Recall: 0.990851058645562
F1 Score: 0.9908696964594318
Accuracy: 0.9877
Precision: 0.9876437716789704
Recall: 0.9875997545695198
F1 Score: 0.9875955213171916


In [ ]:
from torchvision.models import resnet18,ResNet18_Weights
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torch import nn,optim
import torch
from torchvision.models import resnet18,ResNet18_Weights
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
transform = transforms.Compose([transforms.Grayscale(num_output_channels=3),
                                transforms.Resize((224, 224)),
                                  transforms.ToTensor()])

train_dataset = datasets.MNIST(root='../../../data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='../../../data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)
for param in model.parameters():
    param.requires_grad = False  # Fine-tune all layers
model.fc = nn.Linear(model.fc.in_features, 10)


optimizer = optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()
num_epochs=10
for epoch in range(1,num_epochs+1):
    model.train()
    total_loss = 0
    for batch_idx, (data, target) in enumerate(train_loader):
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch {epoch}/{num_epochs}, Loss: {total_loss/len(train_loader)}')
model.eval()
with torch.no_grad():
    actual = []
    predicted = []
    for data, target in train_loader:
        output = model(data)
        _, pred = torch.max(output.data, 1)
        predicted.extend(pred)
        actual.extend(target)
    print(f'Accuracy: {accuracy_score(actual, predicted)}')
    print(f'Precision: {precision_score(actual, predicted, average="macro")}')
    print(f'Recall: {recall_score(actual, predicted, average="macro")}')    
    print(f'F1 Score: {f1_score(actual, predicted, average="macro")}')

with torch.no_grad():
    actual = []
    predicted = []
    for data, target in test_loader:
        output = model(data)
        _, pred = torch.max(output.data, 1)
        predicted.extend(pred)
        actual.extend(target)
    print(f'Accuracy: {accuracy_score(actual, predicted)}')
    print(f'Precision: {precision_score(actual, predicted, average="macro")}')
    print(f'Recall: {recall_score(actual, predicted, average="macro")}')    
    print(f'F1 Score: {f1_score(actual, predicted, average="macro")}')


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\barat/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:01<00:00, 38.2MB/s]


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torch import nn,optim
import torch
from torchvision.models import alexnet,AlexNet_Weights
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
transform = transforms.Compose([transforms.Grayscale(num_output_channels=3),
                                transforms.Resize((224, 224)),
                                  transforms.ToTensor()])

train_dataset = datasets.MNIST(root='../../../data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='../../../data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
weights = AlexNet_Weights.DEFAULT
model = alexnet(weights=weights)
for param in model.parameters():
    param.requires_grad = False  # Fine-tune all layers
model.classifier[6] = nn.Linear(model.classifier[6].in_features, 10)


optimizer = optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()
num_epochs=10
for epoch in range(1,num_epochs+1):
    model.train()
    total_loss = 0
    for batch_idx, (data, target) in enumerate(train_loader):
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch {epoch}/{num_epochs}, Loss: {total_loss/len(train_loader)}')
model.eval()
with torch.no_grad():
    actual = []
    predicted = []
    for data, target in train_loader:
        output = model(data)
        _, pred = torch.max(output.data, 1)
        predicted.extend(pred)
        actual.extend(target)
    print(f'Accuracy: {accuracy_score(actual, predicted)}')
    print(f'Precision: {precision_score(actual, predicted, average="macro")}')
    print(f'Recall: {recall_score(actual, predicted, average="macro")}')    
    print(f'F1 Score: {f1_score(actual, predicted, average="macro")}')

with torch.no_grad():
    actual = []
    predicted = []
    for data, target in test_loader:
        output = model(data)
        _, pred = torch.max(output.data, 1)
        predicted.extend(pred)
        actual.extend(target)
    print(f'Accuracy: {accuracy_score(actual, predicted)}')
    print(f'Precision: {precision_score(actual, predicted, average="macro")}')
    print(f'Recall: {recall_score(actual, predicted, average="macro")}')    
    print(f'F1 Score: {f1_score(actual, predicted, average="macro")}')
